In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('raw-data/13F_ARK_Raw.csv')
df.head()

,Sym,Issuer Name,Cl,CUSIP,Value ($000),%,Shares,Principal,Option Type
0,TSLA,Tesla Inc,COMMON STOCK,88160R101,"1,052,546",8.2%,"2,831,329",NaN,NaN
1,AMD,Advanced Micro Devices Inc,COMMON STOCK,7903107,"551,875",4.3%,"2,712,854",NaN,NaN
2,CRSP,CRISPR Therapeutics AG,COMMON STOCK,H17182108,"538,189",4.2%,"11,313,623",NaN,NaN
3,SHOP,Shopify Inc,COMMON STOCK,82509L107,"495,553",3.9%,"4,177,652",NaN,NaN
4,PLTR,Palantir Technologies Inc,COMMON STOCK,69608A108,"454,913",3.5%,"3,109,881",NaN,NaN


In [4]:
print(df.columns.tolist())
df = df[['Sym',"Issuer Name", "Cl", '%']]
df.head()

['Sym', 'Issuer Name', 'Cl', 'CUSIP', 'Value ($000)', '%', 'Shares', 'Principal', 'Option Type']


,Sym,Issuer Name,Cl,%
0,TSLA,Tesla Inc,COMMON STOCK,8.2%
1,AMD,Advanced Micro Devices Inc,COMMON STOCK,4.3%
2,CRSP,CRISPR Therapeutics AG,COMMON STOCK,4.2%
3,SHOP,Shopify Inc,COMMON STOCK,3.9%
4,PLTR,Palantir Technologies Inc,COMMON STOCK,3.5%


In [7]:
def common_stock_filter(df):
    return df[df['Cl']== 'COMMON STOCK']
df_common = common_stock_filter(df)
df_common

,Sym,Issuer Name,Cl,%
0,TSLA,Tesla Inc,COMMON STOCK,8.2%
1,AMD,Advanced Micro Devices Inc,COMMON STOCK,4.3%
2,CRSP,CRISPR Therapeutics AG,COMMON STOCK,4.2%
3,SHOP,Shopify Inc,COMMON STOCK,3.9%
4,PLTR,Palantir Technologies Inc,COMMON STOCK,3.5%
...,...,...,...,...
173,KMT,Kennametal Inc,COMMON STOCK,0.0%
175,AVNT,Avient Corp,COMMON STOCK,0.0%
176,MTRN,Materion Corp,COMMON STOCK,0.0%
177,MMM,3M Co,COMMON STOCK,0.0%


In [5]:
import yfinance as yf
from pykrx import stock
from tqdm import tqdm

In [ ]:
def get_fundamentals(ticker):
    try:
        info = yf.Ticker(ticker).info
        return {
            'Sym':          ticker,
            'rev_growth':   info.get('revenueGrowth'),
            'gross_margin': info.get('grossMargins'),
            'ps_ratio':     info.get('priceToSalesTrailing12Months'),
            'market_cap':   info.get('marketCap'),
            'sector':       info.get('sector'),
        }
    except Exception as e:
        print(ticker, e)
        return None

records = [get_fundamentals(t) for t in tqdm(df_common['Sym'])]
fundamentals = pd.DataFrame([r for r in records if r])

ark_enriched = df_common.merge(fundamentals, on='Sym').dropna()

100%|██████████| 158/158 [01:02<00:00,  2.51it/s]


In [10]:
ark_enriched

,Sym,Issuer Name,Cl,%,rev_growth,gross_margin,ps_ratio,market_cap,sector
0,TSLA,Tesla Inc,COMMON STOCK,8.2%,0.158,0.19065,14.393763,1.408847e+12,Consumer Cyclical
1,AMD,Advanced Micro Devices Inc,COMMON STOCK,4.3%,0.378,0.53060,23.186014,8.684090e+11,Technology
2,CRSP,CRISPR Therapeutics AG,COMMON STOCK,4.2%,0.686,0.00000,1295.679600,5.316173e+09,Healthcare
3,SHOP,Shopify Inc,COMMON STOCK,3.9%,0.343,0.47970,11.713101,1.448442e+11,Technology
4,PLTR,Palantir Technologies Inc,COMMON STOCK,3.5%,0.847,0.84074,49.224937,2.571596e+11,Technology
...,...,...,...,...,...,...,...,...,...
152,JBL,Jabil Inc,COMMON STOCK,0.0%,0.118,0.09226,1.176705,3.952550e+10,Technology
153,KMT,Kennametal Inc,COMMON STOCK,0.0%,0.218,0.32078,1.303026,2.783955e+09,Industrials
154,AVNT,Avient Corp,COMMON STOCK,0.0%,0.025,0.32508,1.034220,3.393276e+09,Basic Materials
155,MTRN,Materion Corp,COMMON STOCK,0.0%,0.308,0.16403,3.059354,5.861858e+09,Basic Materials
